# analyze_exported_npz notebook
Interactive version of `analyze_exported_npz.py`.

- One plotting block per cell.
- Relevant variables are defined at the top of each plotting cell.
- Re-run the data-prep cell after changing settings in `analyze_exported_npz.py` or `analyze_experimental_data.py`.

In [ ]:
from importlib import reload
import sys
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np

try:
    import Experimental_Data.analyze_experimental_data as analysis
    import Experimental_Data.analyze_exported_npz as npz_analysis
except ModuleNotFoundError as exc:
    if getattr(exc, 'name', '') != 'Experimental_Data':
        raise
    current_dir = Path('Experimental_Data').resolve()
    if str(current_dir) not in sys.path:
        sys.path.insert(0, str(current_dir))
    import analyze_experimental_data as analysis
    import analyze_exported_npz as npz_analysis

analysis = reload(analysis)
npz_analysis = reload(npz_analysis)

In [ ]:
# Data prep
PRINT_SUMMARY = True
npz_files = npz_analysis._resolve_npz_files()
entries = [npz_analysis._process_npz(p) for p in npz_files]
colors = plt.cm.tab10(np.linspace(0.0, 1.0, max(len(entries), 2)))
n_rows = len(entries)

if PRINT_SUMMARY:
    for entry in entries:
        s = entry['summary']
        print(
            f"{entry['label']}: Umean={s['umean']:.6f} m/s, Ur={s['ur']:.6f}, "
            f"Cd={s['cd']:.6f}, Ydomfreq={s['ydomfreq']:.6f} +/- {s['ydomfreq_std']:.6f} Hz, "
            f"nt={s['nt']}, dt={s['dt']:.6f} s, Fs={s['fs']:.6f} Hz, "
            f"coeff_norm={s['coeff_norm_mode']}"
        )

print(f'Loaded {len(entries)} entries from {len(npz_files)} npz file(s).')

## Simple Force Builder (NPZ)
Define one or more force models here using loaded `calculated_force` plus optional inertia terms.

This notebook does not carry separate spring channels, so models are based on:
- `base_force_mult * cfy_force_loaded`
- optional inertia term `m * y_ddot`
- `global_sign` on the full expression.

In [ ]:
# Simple force-generation block for NPZ analysis
USE_CUSTOM_FORCE_MODELS = True

FORCE_MODELS = [
    {
        "name": "loaded_force",
        "base_force_mult": +1.0,
        "global_sign": +1.0,
        "include_structural_mass": False,
        "include_added_mass": False,
        "inertia_sign": +1.0,
    },
    {
        "name": "loaded_plus_M",
        "base_force_mult": +1.0,
        "global_sign": +1.0,
        "include_structural_mass": True,
        "include_added_mass": False,
        "inertia_sign": +1.0,
    },
]

ACTIVE_FORCE_MODEL = "loaded_force"
PLOT_FORCE_MODELS = []
FORCE_LINESTYLES = ["-", "--", "-.", ":"]


def _normalize_models(models):
    out = []
    for i, m in enumerate(models):
        name = str(m.get("name", f"force_{i+1}")).strip() or f"force_{i+1}"
        out.append({
            "name": name,
            "base_force_mult": float(m.get("base_force_mult", 1.0)),
            "global_sign": float(m.get("global_sign", 1.0)),
            "include_structural_mass": bool(m.get("include_structural_mass", False)),
            "include_added_mass": bool(m.get("include_added_mass", False)),
            "inertia_sign": float(m.get("inertia_sign", 1.0)),
        })
    names = [m["name"] for m in out]
    if len(set(names)) != len(names):
        raise ValueError("FORCE_MODELS must have unique names.")
    return out


if USE_CUSTOM_FORCE_MODELS:
    models = _normalize_models(FORCE_MODELS)
    model_names = [m["name"] for m in models]
    if ACTIVE_FORCE_MODEL not in model_names:
        raise ValueError(f"ACTIVE_FORCE_MODEL='{ACTIVE_FORCE_MODEL}' not in {model_names}")
    plot_names = model_names if not PLOT_FORCE_MODELS else [str(v) for v in PLOT_FORCE_MODELS]
    unknown = [v for v in plot_names if v not in model_names]
    if unknown:
        raise ValueError(f"Unknown model(s) in PLOT_FORCE_MODELS: {unknown}")

    m_added = float(analysis.ADDED_MASS_COEFF) * 0.25 * np.pi * analysis.RUO * analysis.D * analysis.D * analysis.L

    for entry in entries:
        f_loaded = np.asarray(entry["cfy_force"], dtype=float)
        yacc = np.asarray(entry["yacc"], dtype=float)
        q_ref_vec = np.asarray(entry["q_ref_vec"], dtype=float)

        cfy_models = {}
        cfy_force_models = {}
        for model in models:
            m_inertia = 0.0
            if model["include_structural_mass"]:
                m_inertia += float(analysis.M)
            if model["include_added_mass"]:
                m_inertia += float(m_added)

            f_model = model["global_sign"] * (
                model["base_force_mult"] * f_loaded
                + model["inertia_sign"] * m_inertia * yacc
            )
            cfy_force_models[model["name"]] = np.asarray(f_model, dtype=float)
            if analysis._use_raw_force_signals():
                cfy_models[model["name"]] = np.asarray(f_model, dtype=float)
            else:
                cfy_models[model["name"]] = np.asarray(f_model, dtype=float) / q_ref_vec

        entry["cfy_models"] = cfy_models
        entry["cfy_force_models"] = cfy_force_models
        entry["cfy_model_names"] = model_names
        entry["cfy_model_plot_names"] = plot_names
        entry["cfy_model_active"] = ACTIVE_FORCE_MODEL

        entry["cfy"] = np.asarray(cfy_models[ACTIVE_FORCE_MODEL], dtype=float)
        entry["cfy_force"] = np.asarray(cfy_force_models[ACTIVE_FORCE_MODEL], dtype=float)

        fs = float(entry["summary"]["fs"])
        spc_cfy, f_cfy, n_cfy = analysis._spec(entry["cfy"], fs)
        if analysis.NORMALIZE_SPECTRA:
            spc_cfy = analysis._normalize_spectrum(spc_cfy, eps=analysis.SPECTRUM_NORM_EPS)
        entry["sp_cfy"] = (f_cfy, spc_cfy, n_cfy)

        y_nd = np.asarray(entry["y_nd"], dtype=float)
        ydomfreq = float(entry["summary"]["ydomfreq"])
        entry["summary"]["phase_cfy_y_deg"] = analysis._phase_lag_deg_at_frequency(
            y_nd,
            np.asarray(entry["cfy"], dtype=float),
            fs=fs,
            target_hz=ydomfreq,
        )

    print("Custom NPZ force models applied:", model_names)
    print("Active model:", ACTIVE_FORCE_MODEL)
else:
    for entry in entries:
        entry.setdefault("cfy_models", {"default": np.asarray(entry["cfy"], dtype=float)})
        entry.setdefault("cfy_force_models", {"default": np.asarray(entry["cfy_force"], dtype=float)})
        entry.setdefault("cfy_model_names", ["default"])
        entry.setdefault("cfy_model_plot_names", ["default"])
        entry.setdefault("cfy_model_active", "default")


In [ ]:
# Custom force-model overlay (all selected models together)
FORCE_OVERLAY_WIDTH = 12
FORCE_OVERLAY_ROW_HEIGHT = 1.8
FORCE_OVERLAY_MIN_HEIGHT = 3.0

fig_force, axes_force = plt.subplots(
    n_rows,
    1,
    figsize=(FORCE_OVERLAY_WIDTH, max(FORCE_OVERLAY_ROW_HEIGHT * n_rows, FORCE_OVERLAY_MIN_HEIGHT)),
    sharex=True,
)
axes_force = np.atleast_1d(axes_force)

for i, entry in enumerate(entries):
    ax = axes_force[i]
    base_color = colors[i % len(colors)]
    label = str(entry["label"])
    t = np.asarray(entry["time_plot"], dtype=float)
    plot_names = list(entry.get("cfy_model_plot_names", entry.get("cfy_model_names", [])))
    cfy_models = entry.get("cfy_models", {})

    for j, name in enumerate(plot_names):
        if name not in cfy_models:
            continue
        ls = FORCE_LINESTYLES[j % len(FORCE_LINESTYLES)]
        ax.plot(t, np.asarray(cfy_models[name], dtype=float), color=base_color, linestyle=ls, linewidth=1.2, label=name)

    ax.grid(True)
    ax.set_ylabel(f"{label}\n{analysis._cf_label()}")
    if len(plot_names) > 1:
        ax.legend(loc="best", fontsize="small")

axes_force[0].set_title("Configured CF force models (overlay)")
axes_force[-1].set_xlabel("Time (s)")
fig_force.tight_layout()
plt.show()


In [ ]:
# Figure 1: full timeseries
FIG1_WIDTH = 17
FIG1_ROW_HEIGHT = 1.1
FIG1_MIN_HEIGHT = 2.0

fig1, axes1 = plt.subplots(
    n_rows,
    5,
    figsize=(FIG1_WIDTH, max(FIG1_ROW_HEIGHT * n_rows, FIG1_MIN_HEIGHT)),
    sharex='col',
)
axes1 = np.atleast_2d(axes1)
col_titles_full = [
    'Measured reduced velocity U_r',
    'Measured CF displacement y/D (mean removed)',
    'Measured CF acceleration y_ddot',
    f"Measured {analysis._drag_name()}",
    f"Measured {analysis._cf_name()} ({analysis._cf_force_mode_label()})",
]
for j, title in enumerate(col_titles_full):
    axes1[0, j].set_title(title)

for i, entry in enumerate(entries):
    color = colors[i % len(colors)]
    label = str(entry['label'])
    t = np.asarray(entry['time_plot'])
    ax_ur, ax_y, ax_acc, ax_cd, ax_cf = axes1[i, 0], axes1[i, 1], axes1[i, 2], axes1[i, 3], axes1[i, 4]
    ax_ur.plot(t, np.asarray(entry['ur_inst']), color=color)
    ax_y.plot(t, np.asarray(entry['y_nd']), color=color)
    ax_acc.plot(t, np.asarray(entry['yacc']), color=color)
    ax_cd.plot(t, np.asarray(entry['cdrag']), color=color)
    ax_cf.plot(t, np.asarray(entry['cfy']), color=color)
    for ax in (ax_ur, ax_y, ax_acc, ax_cd, ax_cf):
        ax.grid(True)
    ax_ur.set_ylabel(f"{label}\nU_r (-)")
    ax_y.set_ylabel('y/D (-)')
    ax_acc.set_ylabel('y_ddot (m/s^2)')
    ax_cd.set_ylabel(analysis._drag_label())
    ax_cf.set_ylabel(analysis._cf_label())

for j in range(5):
    axes1[-1, j].set_xlabel('Time (s)')
fig1.tight_layout()
plt.show()

In [ ]:
# Figure 3: phase + spectra overlay
FIG3_SIZE = (11, 8)
SPECTRUM_XMIN = 0.1
SPECTRUM_XMAX = float(analysis.SPECTRUM_PLOT_MAX_HZ)

fig3, axes3 = plt.subplots(2, 2, figsize=FIG3_SIZE)
ax_phase = axes3[0, 0]
for i, entry in enumerate(entries):
    color = colors[i % len(colors)]
    label = str(entry['label'])
    y_nd = np.asarray(entry['y_nd'])
    yvel = np.asarray(entry['yvel'])
    ax_phase.plot(y_nd, yvel, color=color, alpha=0.35, label=label)
    ax_phase.scatter(y_nd[0], yvel[0], color=color, s=22, zorder=3)
ax_phase.grid(True)
ax_phase.set_xlabel('CF displacement y/D (mean removed, -)')
ax_phase.set_ylabel('CF velocity dy/dt (m/s)')
ax_phase.set_title('CF phase diagram')
npz_analysis._add_legends([ax_phase])

ax_fy = axes3[0, 1]
ax_fd = axes3[1, 0]
ax_ur = axes3[1, 1]
for i, entry in enumerate(entries):
    color = colors[i % len(colors)]
    label = str(entry['label'])
    f_cfy, sp_cfy, n_cfy = entry['sp_cfy']
    f_cdrag, sp_cdrag, n_cdrag = entry['sp_cdrag']
    f_ur, sp_ur, n_ur = entry['sp_ur']
    ax_fy.plot(np.asarray(f_cfy)[: int(n_cfy)], np.asarray(sp_cfy)[: int(n_cfy)], color=color, label=label)
    ax_fd.plot(np.asarray(f_cdrag)[: int(n_cdrag)], np.asarray(sp_cdrag)[: int(n_cdrag)], color=color, label=label)
    ax_ur.plot(np.asarray(f_ur)[: int(n_ur)], np.asarray(sp_ur)[: int(n_ur)], color=color, label=label)
ax_fy.set_xlim(SPECTRUM_XMIN, SPECTRUM_XMAX)
ax_fy.grid(True)
ax_fy.set_xlabel('Frequency (Hz)')
ax_fy.set_ylabel('Normalized spectrum' if analysis.NORMALIZE_SPECTRA else 'Spectrum')
ax_fy.set_title(f"{analysis._cf_name()} spectrum ({analysis._cf_force_mode_label()})")
ax_fd.set_xlim(SPECTRUM_XMIN, SPECTRUM_XMAX)
ax_fd.grid(True)
ax_fd.set_xlabel('Frequency (Hz)')
ax_fd.set_ylabel('Normalized spectrum' if analysis.NORMALIZE_SPECTRA else 'Spectrum')
ax_fd.set_title(f"{analysis._drag_name()} spectrum")
ax_ur.set_xlim(SPECTRUM_XMIN, SPECTRUM_XMAX)
ax_ur.grid(True)
ax_ur.set_xlabel('Frequency (Hz)')
ax_ur.set_ylabel('Normalized spectrum' if analysis.NORMALIZE_SPECTRA else 'Spectrum')
ax_ur.set_title('Reduced velocity spectrum')
npz_analysis._add_legends([ax_fy, ax_fd, ax_ur])
fig3.tight_layout()
plt.show()

In [ ]:
# Figure 4: first N seconds
FIG4_WIDTH = 17
FIG4_ROW_HEIGHT = 1.0
FIG4_MIN_HEIGHT = 2.0
FIG4_FIRST_WINDOW_SECONDS = float(npz_analysis.FIRST_WINDOW_SECONDS)

fig4, axes4 = plt.subplots(
    n_rows,
    5,
    figsize=(FIG4_WIDTH, max(FIG4_ROW_HEIGHT * n_rows, FIG4_MIN_HEIGHT)),
    sharex='col',
)
axes4 = np.atleast_2d(axes4)
col_titles = [
    f"Reduced velocity U_r (first {FIG4_FIRST_WINDOW_SECONDS:g} s)",
    f"CF displacement y/D (mean removed, first {FIG4_FIRST_WINDOW_SECONDS:g} s)",
    f"CF acceleration y_ddot (first {FIG4_FIRST_WINDOW_SECONDS:g} s)",
    f"{analysis._drag_name()} (first {FIG4_FIRST_WINDOW_SECONDS:g} s)",
    f"{analysis._cf_name()} ({analysis._cf_force_mode_label()}, first {FIG4_FIRST_WINDOW_SECONDS:g} s)",
]
for j, title in enumerate(col_titles):
    axes4[0, j].set_title(title)

for i, entry in enumerate(entries):
    color = colors[i % len(colors)]
    label = str(entry['label'])
    t_full = np.asarray(entry['time_plot'])
    mask = t_full <= FIG4_FIRST_WINDOW_SECONDS
    ax_ur, ax_y, ax_acc, ax_cd, ax_cf = axes4[i, 0], axes4[i, 1], axes4[i, 2], axes4[i, 3], axes4[i, 4]
    if np.any(mask):
        t = t_full[mask]
        ax_ur.plot(t, np.asarray(entry['ur_inst'])[mask], color=color)
        ax_y.plot(t, np.asarray(entry['y_nd'])[mask], color=color)
        ax_acc.plot(t, np.asarray(entry['yacc'])[mask], color=color)
        ax_cd.plot(t, np.asarray(entry['cdrag'])[mask], color=color)
        ax_cf.plot(t, np.asarray(entry['cfy'])[mask], color=color)
    for ax in (ax_ur, ax_y, ax_acc, ax_cd, ax_cf):
        ax.grid(True)
    ax_ur.set_ylabel(f"{label}\nU_r (-)")
    ax_y.set_ylabel('y/D (-)')
    ax_acc.set_ylabel('y_ddot (m/s^2)')
    ax_cd.set_ylabel(analysis._drag_label())
    ax_cf.set_ylabel(analysis._cf_label())

for j in range(5):
    axes4[-1, j].set_xlabel('Time (s)')
fig4.tight_layout()
plt.show()

In [ ]:
# Figure 5: summary trends vs mean reduced velocity (box plots per U_r)
FIG5_SIZE = (12, 4.8)
REF_CA_FOR_FN_LINE = float(analysis.REF_CA_FOR_FN_LINE)
ANNOTATE_POINTS = True

# Frequency sampling settings (time-window control).
FREQ_WINDOW_SECONDS = 4.0
FREQ_STEP_SECONDS = 0.75
MIN_WINDOW_POINTS = 64

# Box-plot styling.
BOXPLOT_SHOW_FLIERS = False
BOXPLOT_FACE_ALPHA = 0.30
BOXPLOT_WIDTH_MODE = 'auto'    # 'auto' or numeric width in U_r units
BOXPLOT_MIN_WIDTH = 0.20
BOXPLOT_MAX_WIDTH = 0.70
BOXPLOT_WIDTH_SCALE = 0.35
PLOT_MEAN_LINE = True

entries_sorted = sorted(entries, key=lambda e: float(e['summary']['ur']))
labels_sorted = [str(e['label']) for e in entries_sorted]

fig5, axes5 = plt.subplots(1, 2, figsize=FIG5_SIZE)
ax_f, ax_a = axes5

m_added_ref = REF_CA_FOR_FN_LINE * 0.25 * np.pi * analysis.RUO * analysis.D * analysis.D * analysis.L
f_n_ref = (1.0 / (2.0 * np.pi)) * np.sqrt(analysis.K / (analysis.M + m_added_ref))


def _window_length_from_seconds(n: int, fs: float, window_seconds: float) -> int:
    if np.isfinite(fs) and fs > 0.0 and np.isfinite(window_seconds) and window_seconds > 0.0:
        win_len = int(round(float(window_seconds) * float(fs)))
    else:
        win_len = n
    return int(max(int(MIN_WINDOW_POINTS), min(n, win_len)))


def _step_points_from_seconds(fs: float, step_seconds: float, win_len: int) -> int:
    if np.isfinite(fs) and fs > 0.0 and np.isfinite(step_seconds) and step_seconds > 0.0:
        step = int(round(float(step_seconds) * float(fs)))
    else:
        step = max(1, win_len // 2)
    return int(max(1, step))


def _window_starts(n: int, win_len: int, step_points: int) -> np.ndarray:
    if n <= win_len:
        return np.array([0], dtype=int)

    starts = np.arange(0, n - win_len + 1, int(max(1, step_points)), dtype=int)
    if starts.size == 0:
        starts = np.array([0], dtype=int)
    tail = n - win_len
    if int(starts[-1]) != int(tail):
        starts = np.append(starts, int(tail))
    return np.unique(starts)


def _box_width(positions: np.ndarray) -> float:
    if np.isscalar(BOXPLOT_WIDTH_MODE) and str(BOXPLOT_WIDTH_MODE).lower() != 'auto':
        return float(BOXPLOT_WIDTH_MODE)
    x = np.asarray(positions, dtype=float)
    if x.size < 2:
        return float(BOXPLOT_MIN_WIDTH)
    diffs = np.diff(np.sort(np.unique(x)))
    diffs = diffs[np.isfinite(diffs) & (diffs > 0)]
    if diffs.size == 0:
        return float(BOXPLOT_MIN_WIDTH)
    w = float(BOXPLOT_WIDTH_SCALE) * float(np.median(diffs))
    return float(np.clip(w, BOXPLOT_MIN_WIDTH, BOXPLOT_MAX_WIDTH))


def _dominant_freq_subbin(freq: np.ndarray, spec: np.ndarray, *, fmin: float, fmax: float) -> float:
    f = np.asarray(freq, dtype=float).reshape(-1)
    s = np.nan_to_num(np.asarray(spec, dtype=float).reshape(-1), nan=0.0, posinf=0.0, neginf=0.0)

    band = (f >= float(fmin)) & (f <= float(fmax))
    if np.any(band):
        fb = f[band]
        sb = s[band]
    else:
        pos = f > 0.0
        if not np.any(pos):
            return float('nan')
        fb = f[pos]
        sb = s[pos]

    if fb.size == 0:
        return float('nan')

    k = int(np.argmax(sb))
    f_peak = float(fb[k])

    # Parabolic refinement around the spectral peak to reduce FFT-bin locking.
    if 0 < k < (fb.size - 1):
        y1 = float(sb[k - 1])
        y2 = float(sb[k])
        y3 = float(sb[k + 1])
        denom = (y1 - 2.0 * y2 + y3)
        if np.isfinite(denom) and abs(denom) > 1e-20:
            delta = 0.5 * (y1 - y3) / denom
            delta = float(np.clip(delta, -1.0, 1.0))
            step_l = float(fb[k] - fb[k - 1])
            step_r = float(fb[k + 1] - fb[k])
            step = 0.5 * (step_l + step_r)
            if np.isfinite(step) and step > 0.0:
                f_peak = float(fb[k] + delta * step)

    return f_peak


def _amplitude_samples_from_peaks(y_nd: np.ndarray) -> np.ndarray:
    y = np.asarray(y_nd, dtype=float).reshape(-1)
    finite = np.isfinite(y)
    y = y[finite]
    if y.size == 0:
        return np.asarray([], dtype=float)

    y_centered = y - float(np.mean(y))
    peaks, _ = analysis.find_peaks(y_centered)
    troughs, _ = analysis.find_peaks(-y_centered)

    amp_samples = np.concatenate([np.abs(y_centered[peaks]), np.abs(y_centered[troughs])])
    if amp_samples.size < 4:
        amp_samples = np.abs(y_centered)

    amp_samples = amp_samples[np.isfinite(amp_samples)]
    return np.asarray(amp_samples, dtype=float)


def _case_trend_samples(entry: dict):
    summary = entry.get('summary', {})
    y_raw = np.asarray(entry.get('y', []), dtype=float)
    y_nd = np.asarray(entry.get('y_nd', []), dtype=float)
    fs = float(summary.get('fs', np.nan))

    n = int(min(y_raw.size, y_nd.size))
    if n < 16:
        f_fb = float(summary.get('ydomfreq', np.nan))
        a_fb = float(summary.get('amp_mean', np.nan))
        f_arr = np.asarray([f_fb], dtype=float) if np.isfinite(f_fb) else np.asarray([], dtype=float)
        a_arr = np.asarray([a_fb], dtype=float) if np.isfinite(a_fb) else np.asarray([], dtype=float)
        return f_arr, a_arr

    y_raw = y_raw[:n]
    y_nd = y_nd[:n]

    # Frequency distribution: windowed dominant-frequency samples.
    win_len_f = _window_length_from_seconds(n, fs=fs, window_seconds=FREQ_WINDOW_SECONDS)
    step_f = _step_points_from_seconds(fs=fs, step_seconds=FREQ_STEP_SECONDS, win_len=win_len_f)
    starts_f = _window_starts(n, win_len_f, step_f)

    freq_samples = []
    for s in starts_f:
        yw = np.asarray(y_raw[s:s + win_len_f], dtype=float)
        try:
            spec_w, freq_w, _ = analysis._spec(yw, fs)
            if analysis.NORMALIZE_SPECTRA:
                spec_w = analysis._normalize_spectrum(spec_w, eps=analysis.SPECTRUM_NORM_EPS)
            f_dom = _dominant_freq_subbin(
                np.asarray(freq_w, dtype=float),
                np.asarray(spec_w, dtype=float),
                fmin=float(analysis.DOM_FREQ_MIN_HZ),
                fmax=float(analysis.DOM_FREQ_MAX_HZ),
            )
        except Exception:
            f_dom = np.nan
        if np.isfinite(f_dom):
            freq_samples.append(float(f_dom))

    # Amplitude distribution: direct peak/trough samples over full case.
    amp_samples = _amplitude_samples_from_peaks(y_nd)

    if not freq_samples:
        f_fb = float(summary.get('ydomfreq', np.nan))
        if np.isfinite(f_fb):
            freq_samples = [f_fb]
    if amp_samples.size == 0:
        a_fb = float(summary.get('amp_mean', np.nan))
        if np.isfinite(a_fb):
            amp_samples = np.asarray([a_fb], dtype=float)

    return np.asarray(freq_samples, dtype=float), np.asarray(amp_samples, dtype=float)


ur_cases = []
labels_cases = []
freq_boxes = []
amp_boxes = []
freq_mean = []
amp_mean = []

for entry, label in zip(entries_sorted, labels_sorted):
    ur_val = float(entry.get('summary', {}).get('ur', np.nan))
    if not np.isfinite(ur_val):
        continue

    f_samp, a_samp = _case_trend_samples(entry)
    if f_samp.size == 0 or a_samp.size == 0:
        continue

    ur_cases.append(ur_val)
    labels_cases.append(label)
    freq_boxes.append(f_samp)
    amp_boxes.append(a_samp)
    freq_mean.append(float(np.nanmean(f_samp)))
    amp_mean.append(float(np.nanmean(a_samp)))

if ur_cases:
    ur_cases = np.asarray(ur_cases, dtype=float)
    freq_mean = np.asarray(freq_mean, dtype=float)
    amp_mean = np.asarray(amp_mean, dtype=float)

    order = np.argsort(ur_cases)
    ur_cases = ur_cases[order]
    labels_cases = [labels_cases[idx] for idx in order]
    freq_boxes = [freq_boxes[idx] for idx in order]
    amp_boxes = [amp_boxes[idx] for idx in order]
    freq_mean = freq_mean[order]
    amp_mean = amp_mean[order]

    width = _box_width(ur_cases)

    bp_f = ax_f.boxplot(
        freq_boxes,
        positions=ur_cases,
        widths=width,
        patch_artist=True,
        showfliers=bool(BOXPLOT_SHOW_FLIERS),
        manage_ticks=False,
    )
    for patch in bp_f['boxes']:
        patch.set_facecolor('tab:blue')
        patch.set_alpha(BOXPLOT_FACE_ALPHA)
        patch.set_edgecolor('tab:blue')
        patch.set_linewidth(1.0)
    for key in ('whiskers', 'caps', 'medians'):
        for artist in bp_f[key]:
            artist.set_color('tab:blue')
            artist.set_linewidth(1.1)

    bp_a = ax_a.boxplot(
        amp_boxes,
        positions=ur_cases,
        widths=width,
        patch_artist=True,
        showfliers=bool(BOXPLOT_SHOW_FLIERS),
        manage_ticks=False,
    )
    for patch in bp_a['boxes']:
        patch.set_facecolor('tab:orange')
        patch.set_alpha(BOXPLOT_FACE_ALPHA)
        patch.set_edgecolor('tab:orange')
        patch.set_linewidth(1.0)
    for key in ('whiskers', 'caps', 'medians'):
        for artist in bp_a[key]:
            artist.set_color('tab:orange')
            artist.set_linewidth(1.1)

    if PLOT_MEAN_LINE:
        ax_f.plot(ur_cases, freq_mean, color='tab:blue', linewidth=1.3, alpha=0.95, label='Windowed mean')

        ax_a.plot(ur_cases, amp_mean, color='tab:orange', linewidth=1.3, alpha=0.95, label='Peak-sample mean')

    if ANNOTATE_POINTS:
        for x, yv, name in zip(ur_cases, freq_mean, labels_cases):
            ax_f.annotate(name, (x, yv), textcoords='offset points', xytext=(4, 4), fontsize=8)

ax_f.axhline(
    f_n_ref,
    color='black',
    linewidth=1.5,
    linestyle='-',
    label=f"Natural frequency (C_a={REF_CA_FOR_FN_LINE:.1f})",
)
ax_f.grid(True)
ax_f.set_xlabel('Mean reduced velocity U_r (-)')
ax_f.set_ylabel('Dominant CF oscillation frequency (Hz)')
ax_f.set_title('Oscillation frequency distribution vs mean reduced velocity')
ax_f.legend(loc='best', fontsize='small')

ax_a.grid(True)
ax_a.set_xlabel('Mean reduced velocity U_r (-)')
ax_a.set_ylabel('Displacement amplitude |y/D| (-)')
ax_a.set_title('Displacement amplitude (peak samples) vs mean reduced velocity')
if PLOT_MEAN_LINE:
    ax_a.legend(loc='best', fontsize='small')

fig5.tight_layout()
plt.show()


In [ ]:
# Figure 6: phase-binned mean phase portrait
FIG6_SIZE = (14, 4.8)
PHASE_MEAN_BINS = int(analysis.PHASE_MEAN_BINS)
PHASE_MIN_SAMPLES_PER_BIN = int(analysis.PHASE_MIN_SAMPLES_PER_BIN)

fig6, axes6 = plt.subplots(1, 3, figsize=FIG6_SIZE)
ax_phase_mean, ax_y_phase, ax_v_phase = axes6

for i, entry in enumerate(entries):
    color = colors[i % len(colors)]
    label = str(entry['label'])
    try:
        phase_stats = analysis._phase_binned_phase_portrait(
            np.asarray(entry['y_nd']),
            np.asarray(entry['yvel']),
            n_bins=PHASE_MEAN_BINS,
            min_samples_per_bin=PHASE_MIN_SAMPLES_PER_BIN,
        )
    except ValueError as exc:
        warnings.warn(f"{label}: skipped mean phase plot ({exc})")
        continue

    theta = np.asarray(phase_stats['theta'], dtype=float)
    mean_y = np.asarray(phase_stats['mean_y'], dtype=float)
    mean_v = np.asarray(phase_stats['mean_v'], dtype=float)
    std_y = np.asarray(phase_stats['std_y'], dtype=float)
    std_v = np.asarray(phase_stats['std_v'], dtype=float)
    upper_y = np.asarray(phase_stats['upper_y'], dtype=float)
    upper_v = np.asarray(phase_stats['upper_v'], dtype=float)
    lower_y = np.asarray(phase_stats['lower_y'], dtype=float)
    lower_v = np.asarray(phase_stats['lower_v'], dtype=float)

    poly_x = np.concatenate([upper_y, lower_y[::-1]])
    poly_y = np.concatenate([upper_v, lower_v[::-1]])
    ax_phase_mean.fill(poly_x, poly_y, color=color, alpha=0.28, edgecolor='none')
    ax_phase_mean.plot(mean_y, mean_v, color=color, linewidth=1.8, label=label)

    phase_norm = theta / (2.0 * np.pi)
    ax_y_phase.fill_between(phase_norm, mean_y - std_y, mean_y + std_y, color=color, alpha=0.28, edgecolor='none')
    ax_y_phase.plot(phase_norm, mean_y, color=color, linewidth=1.8, label=label)

    ax_v_phase.fill_between(phase_norm, mean_v - std_v, mean_v + std_v, color=color, alpha=0.28, edgecolor='none')
    ax_v_phase.plot(phase_norm, mean_v, color=color, linewidth=1.8, label=label)

ax_phase_mean.grid(True)
ax_phase_mean.set_xlabel('CF displacement y/D (mean removed, -)')
ax_phase_mean.set_ylabel('CF velocity dy/dt (m/s)')
ax_phase_mean.set_title('Mean phase portrait with ±1σ tube')
ax_y_phase.grid(True)
ax_y_phase.set_xlabel('Phase / 2π (-)')
ax_y_phase.set_ylabel('CF displacement y/D (mean removed, -)')
ax_y_phase.set_title('Mean displacement vs phase')
ax_v_phase.grid(True)
ax_v_phase.set_xlabel('Phase / 2π (-)')
ax_v_phase.set_ylabel('CF velocity dy/dt (m/s)')
ax_v_phase.set_title('Mean velocity vs phase')
npz_analysis._add_legends([ax_phase_mean])
fig6.tight_layout()
plt.show()

In [ ]:
# Figure 7: hysteresis loops
FIG7_SIZE = (12, 4.8)
HYSTERESIS_ALPHA = 0.55
HYSTERESIS_LINEWIDTH = 1.0

fig7, (ax_h_cf, ax_h_cd) = plt.subplots(1, 2, figsize=FIG7_SIZE)
for i, entry in enumerate(entries):
    color = colors[i % len(colors)]
    label = str(entry['label'])
    y_nd = np.asarray(entry['y_nd'], dtype=float)
    cfy = np.asarray(entry['cfy'], dtype=float)
    cdrag = np.asarray(entry['cdrag'], dtype=float)
    mask_cf = np.isfinite(y_nd) & np.isfinite(cfy)
    mask_cd = np.isfinite(y_nd) & np.isfinite(cdrag)
    ax_h_cf.plot(y_nd[mask_cf], cfy[mask_cf], color=color, alpha=HYSTERESIS_ALPHA, linewidth=HYSTERESIS_LINEWIDTH, label=label)
    ax_h_cd.plot(y_nd[mask_cd], cdrag[mask_cd], color=color, alpha=HYSTERESIS_ALPHA, linewidth=HYSTERESIS_LINEWIDTH, label=label)
ax_h_cf.grid(True)
ax_h_cf.set_xlabel('Displacement y/D (mean removed, -)')
ax_h_cf.set_ylabel(analysis._cf_label())
ax_h_cf.set_title(f"Hysteresis loop: {analysis._cf_name()} vs y/D")
ax_h_cd.grid(True)
ax_h_cd.set_xlabel('Displacement y/D (mean removed, -)')
ax_h_cd.set_ylabel(analysis._drag_label())
ax_h_cd.set_title(f"Hysteresis loop: {analysis._drag_name()} vs y/D")
npz_analysis._add_legends([ax_h_cf, ax_h_cd])
fig7.tight_layout()
plt.show()

In [ ]:
# Figure 8: phase lag vs reduced velocity
FIG8_SIZE = (12, 4.6)
PHASE_YLIM = (-185.0, 185.0)
ANNOTATE_POINTS = True

entries_sorted = sorted(entries, key=lambda e: float(e['summary']['ur']))
ur_vals = np.array([float(e['summary']['ur']) for e in entries_sorted], dtype=float)
labels_sorted = [str(e['label']) for e in entries_sorted]
phase_cfy_vals = np.array([float(e['summary']['phase_cfy_y_deg']) for e in entries_sorted], dtype=float)
phase_cdrag_vals = np.array([float(e['summary']['phase_cdrag_y_deg']) for e in entries_sorted], dtype=float)

fig8, (ax_p_cf, ax_p_cd) = plt.subplots(1, 2, figsize=FIG8_SIZE, sharey=True)

phase_cf_mask = np.isfinite(ur_vals) & np.isfinite(phase_cfy_vals)
ur_phase_cf = ur_vals[phase_cf_mask]
phase_cf = phase_cfy_vals[phase_cf_mask]
labels_phase_cf = [name for name, keep in zip(labels_sorted, phase_cf_mask) if keep]
ax_p_cf.scatter(
    ur_phase_cf,
    phase_cf,
    color='tab:blue',
    s=42,
    label=f"Phase({analysis._cf_name()}) - Phase(y)",
)
ax_p_cf.axhline(0.0, color='black', linewidth=1.0, alpha=0.7)
if ANNOTATE_POINTS:
    for x, yv, name in zip(ur_phase_cf, phase_cf, labels_phase_cf):
        ax_p_cf.annotate(name, (x, yv), textcoords='offset points', xytext=(4, 4), fontsize=8)
ax_p_cf.grid(True)
ax_p_cf.set_xlabel('Mean reduced velocity U_r (-)')
ax_p_cf.set_ylabel('Phase lag (deg)')
ax_p_cf.set_title(f"Phase lag: {analysis._cf_name()} relative to y")
ax_p_cf.set_ylim(*PHASE_YLIM)
ax_p_cf.legend(loc='best', fontsize='small')

phase_cd_mask = np.isfinite(ur_vals) & np.isfinite(phase_cdrag_vals)
ur_phase_cd = ur_vals[phase_cd_mask]
phase_cd = phase_cdrag_vals[phase_cd_mask]
labels_phase_cd = [name for name, keep in zip(labels_sorted, phase_cd_mask) if keep]
ax_p_cd.scatter(
    ur_phase_cd,
    phase_cd,
    color='tab:orange',
    s=42,
    label=f"Phase({analysis._drag_name()}) - Phase(y)",
)
ax_p_cd.axhline(0.0, color='black', linewidth=1.0, alpha=0.7)
if ANNOTATE_POINTS:
    for x, yv, name in zip(ur_phase_cd, phase_cd, labels_phase_cd):
        ax_p_cd.annotate(name, (x, yv), textcoords='offset points', xytext=(4, 4), fontsize=8)
ax_p_cd.grid(True)
ax_p_cd.set_xlabel('Mean reduced velocity U_r (-)')
ax_p_cd.set_title(f"Phase lag: {analysis._drag_name()} relative to y")
ax_p_cd.legend(loc='best', fontsize='small')
fig8.tight_layout()
plt.show()